# Fully Corrected Neural Network Assignment Notebook

In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import confusion_matrix, classification_report

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense


## Load Dataset

In [ ]:

df = pd.read_csv("customer_churn_nn.csv")

print(df.head())


## Dataset Exploration

In [ ]:

print("Dataset Shape:")
print(df.shape)

print("\nColumns:")
print(df.columns.tolist())

print("\nData Types:")
print(df.dtypes)

print("\nMissing Values:")
print(df.isnull().sum())


## Remove Missing Values

In [ ]:

df = df.dropna()


## Automatically Select Target Column

In [ ]:

target_column = df.columns[-1]

print("Target Column:", target_column)


## Target Variable Distribution

In [ ]:

sns.countplot(x=target_column, data=df)

plt.title("Target Variable Distribution")

plt.show()


## Split Features and Target

In [ ]:

X = df.drop(columns=[target_column])

y = df[target_column]


## Convert Categorical Columns Using get_dummies

In [ ]:

X = pd.get_dummies(X, drop_first=True)

print(X.head())

print(X.dtypes)


## Encode Target Variable

In [ ]:

if y.dtype == 'object':
    encoder = LabelEncoder()
    y = encoder.fit_transform(y)

print(np.unique(y))


## Train Test Split

In [ ]:

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)


## Convert All Data to Float

In [ ]:

X_train = X_train.astype(float)

X_test = X_test.astype(float)


## Feature Scaling

In [ ]:

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)

X_test = scaler.transform(X_test)

print(X_train.shape)

print(X_test.shape)


## Build Neural Network

In [ ]:

model = Sequential()

model.add(Dense(16, activation='relu', input_shape=(X_train.shape[1],)))

model.add(Dense(8, activation='relu'))

model.add(Dense(1, activation='sigmoid'))


## Compile Model

In [ ]:

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)


## Train Model

In [ ]:

history = model.fit(
    X_train,
    y_train,
    epochs=20,
    batch_size=32,
    validation_split=0.2
)


## Evaluate Model

In [ ]:

loss, accuracy = model.evaluate(X_test, y_test)

print("Test Loss:", loss)

print("Test Accuracy:", accuracy)


## Predictions

In [ ]:

y_pred_prob = model.predict(X_test)

y_pred = (y_pred_prob > 0.5).astype(int)

y_pred = y_pred.flatten()

print(y_pred[:10])


## Confusion Matrix

In [ ]:

cm = confusion_matrix(y_test, y_pred)

sns.heatmap(cm, annot=True, fmt='d')

plt.title("Confusion Matrix")

plt.xlabel("Predicted")

plt.ylabel("Actual")

plt.show()


## Classification Report

In [ ]:

print(classification_report(y_test, y_pred))


## Hyperparameter Experiments

In [ ]:

results = []

def run_experiment(neurons, epochs, batch_size):

    model = Sequential()

    model.add(Dense(neurons, activation='relu', input_shape=(X_train.shape[1],)))

    model.add(Dense(1, activation='sigmoid'))

    model.compile(
        optimizer='adam',
        loss='binary_crossentropy',
        metrics=['accuracy']
    )

    model.fit(
        X_train,
        y_train,
        epochs=epochs,
        batch_size=batch_size,
        verbose=0
    )

    loss, accuracy = model.evaluate(X_test, y_test, verbose=0)

    results.append([neurons, epochs, batch_size, accuracy])


In [ ]:

run_experiment(8, 10, 32)

run_experiment(16, 20, 32)

run_experiment(32, 30, 64)


In [ ]:

comparison_df = pd.DataFrame(results, columns=[
    'Neurons',
    'Epochs',
    'Batch Size',
    'Accuracy'
])

print(comparison_df)

comparison_df.to_csv(
    "model_comparison_table.csv",
    index=False
)



# Final Reflection

## Role of Weights and Biases
Weights help the model learn patterns from input data, while biases improve flexibility in learning.

## Importance of Activation Functions
Activation functions introduce non-linearity and allow neural networks to solve complex problems.

## Learning Rate Effects
A high learning rate may overshoot the best solution, while a low learning rate slows training.

## Underfitting and Overfitting
Underfitting happens when the model learns too little. Overfitting happens when the model memorizes training data.
